# 🤖 AutoML Pilot - Pro Training Template
Use this Colab notebook to run advanced AutoML (PyCaret) on your dataset with automated reporting and EDA.

In [ ]:
# @title 1. Install Dependencies
!pip install -q pycaret[full] ydata-profiling pandas jinja2 shap
print('✅ Dependencies installed!')

In [ ]:
# @title 2. Load Dataset
import pandas as pd
import os
from google.colab import files

uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)
print(f'✅ Loaded {filename}: {df.shape[0]} rows')

In [ ]:
# @title 3. Automated EDA
run_eda = True # @param {type:"boolean"}
if run_eda:
    from ydata_profiling import ProfileReport
    profile = ProfileReport(df, title="Profiling Report", explorative=True)
    profile.to_notebook_iframe()

In [ ]:
# @title 4. AutoML Setup & Training
from pycaret.classification import setup as cls_setup, compare_models as cls_compare, pull as cls_pull, save_model as cls_save, finalize_model as cls_finalize, interpret_model
from pycaret.regression import setup as reg_setup, compare_models as reg_compare, pull as reg_pull, save_model as reg_save, finalize_model as reg_finalize, interpret_model as interpret_reg_model

target_column = 'target' # @param {type:"string"}
task_type = 'classification' # @param ["classification", "regression"]

if task_type == 'classification':
    s = cls_setup(data=df, target=target_column, session_id=123, verbose=False)
    best_model = cls_compare()
    leaderboard = cls_pull()
    final_model = cls_finalize(best_model)
    cls_save(final_model, 'best_model')
    try:
        interpret_model(final_model)
    except:
        pass
else:
    s = reg_setup(data=df, target=target_column, session_id=123, verbose=False)
    best_model = reg_compare()
    leaderboard = reg_pull()
    final_model = reg_finalize(best_model)
    reg_save(final_model, 'best_model')
    try:
        interpret_reg_model(final_model)
    except:
        pass

display(leaderboard.head())

In [ ]:
# @title 5. Automated Email Reporting
import smtplib, ssl
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from jinja2 import Template

recipient_email = '' # @param {type:"string"}
sender_email = '' # @param {type:"string"}
sender_password = '' # @param {type:"string"}

html_template = """
<html>
<body style="font-family: Arial, sans-serif; color: #333;">
    <div style="max-width: 600px; margin: auto; padding: 20px; border: 1px solid #ddd; border-radius: 10px;">
        <h2 style="color: #4CAF50; text-align: center;">🚀 AutoML Pilot Pro Training Report</h2>
        <p>Hello,</p>
        <p>Your model training on Google Colab has completed successfully. Here are the top performing models:</p>
        <div style="overflow-x: auto;">
            {{ leaderboard_html }}
        </div>
        <p>You can now download the <code>best_model.pkl</code> and upload it to the AutoML Pilot Deployment tab for instant inference.</p>
        <hr>
        <p style="font-size: 0.8em; color: #777; text-align: center;">Sent via AutoML Pilot Pro Colab Integration</p>
    </div>
</body>
</html>
"""

if recipient_email and sender_email and sender_password:
    msg = MIMEMultipart("alternative")
    msg["Subject"] = f"AutoML Pilot Pro - {task_type.capitalize()} Report"
    msg["From"] = sender_email
    msg["To"] = recipient_email

    template = Template(html_template)
    html_content = template.render(leaderboard_html=leaderboard.head().to_html(classes='table table-striped'))
    
    msg.attach(MIMEText(html_content, "html"))
    context = ssl.create_default_context()
    with smtplib.SMTP_SSL("smtp.gmail.com", 465, context=context) as server:
        server.login(sender_email, sender_password)
        server.sendmail(sender_email, recipient_email, msg.as_string())
    print('✅ Professional email report sent!')

In [ ]:
# @title 6. Download Model
from google.colab import files
if os.path.exists('best_model.pkl'):
    files.download('best_model.pkl')
    print('✅ Model downloaded!')